# Hypothesis Testing in Python: A Guided Tutorial

This notebook is a step-by-step guide to hypothesis testing using three different distributions: normal, binomial, and Poisson.

It links hypothesis testing to the concepts you already know from distributions and confidence intervals (CI). For each distribution, we walk through a worked case study.

## Learning Objectives

At the end of this notebook, you should be able to:

- State null and alternative hypotheses for a business question.
- Select the appropriate hypothesis test based on the distribution of the data.
- Compute test statistics and p-values with scipy and statsmodels.
- Apply a significance level to decide whether to reject the null hypothesis, and interpret that decision in context.

In [ ]:
# Import necessary libraries
import numpy as np
from statsmodels.stats.proportion import proportions_ztest
from scipy.stats import poisson, norm

# Seed for reproducibility
np.random.seed(42)

---
## 1. Introduction to Hypothesis Testing

A hypothesis test asks a simple question: **is the pattern in my sample strong enough to conclude something about the whole population, or could it just be random variation?**

We answer it by assuming the boring explanation (the null hypothesis) is true, then computing how surprising our sample would be under that assumption. That surprise is the **p-value**: the probability of seeing data at least as extreme as ours if the null hypothesis were true. A small p-value means the data is hard to explain by chance alone.

### The steps in every hypothesis test

1. State the null hypothesis (H0) and the alternative hypothesis (H1).
2. Choose a significance level (alpha), typically 0.05.
3. Collect and prepare the data.
4. Choose the appropriate test based on the distribution and sample size.
5. Conduct the test and calculate the test statistic.
6. Determine the p-value.
7. Make a decision: reject or fail to reject the null hypothesis.
8. Interpret the result in the context of the original question.

Two key connections to earlier lessons:

- **Distributions** determine which test is appropriate: the test statistic only has a known distribution if we model the data correctly.
- **Confidence intervals** offer the same information from another angle: a 95% CI that excludes the null value corresponds to rejecting H0 at alpha = 0.05.

### Which test do we use?

The three case studies in this notebook follow one decision: match the test to the distribution of the data.

```mermaid
flowchart TD
    A[What kind of data?] --> B[Continuous measurement]
    A --> C[Successes out of n trials]
    A --> D[Counts of events per interval]
    B --> E[Normal distribution: z-test, population std known]
    C --> F[Binomial distribution: z-test for a proportion]
    D --> G[Poisson distribution: exact test on Poisson probabilities]
```

If the population standard deviation were unknown in the first case, we would use a t-test instead of a z-test. You will meet that variant in the combined guide.

---

## 2. Case Study 1: Normal Distribution

**Problem.** You are a quality control analyst at a factory producing light bulbs. The average lifetime of the light bulbs is expected to be 1000 hours. You take a sample of 50 bulbs and want to test whether the mean lifetime is significantly different from 1000 hours.

### Step 1: Generate the data

We simulate a sample of 50 lifetimes from a normal distribution. In a real analysis this would be your measured data; simulating it lets us know the true population parameters and check the test behaves sensibly.

In [ ]:
sample_size = 50
population_mean = 1000  # Population mean
population_std = 100  # Population standard deviation (known)

# Simulate sample data (assuming a normal distribution)
sample_data = np.random.normal(
    loc=population_mean, scale=population_std, size=sample_size
)

### Step 2: Calculate the sample mean and standard error

The **standard error** measures how much sample means vary from sample to sample: population standard deviation divided by the square root of the sample size. It is the yardstick against which we judge how far our sample mean is from the hypothesised mean.

In [ ]:
sample_mean = np.mean(sample_data)
standard_error = population_std / np.sqrt(sample_size)

### Step 3: State the hypotheses

- H0: mu = 1000 (the mean lifetime is 1000 hours)
- H1: mu != 1000 (the mean lifetime is different from 1000 hours)

The alternative is two-sided: we care about deviations in either direction, shorter-lived or longer-lived bulbs.

### Step 4: Perform the z-test

We use a **z-test** because the population standard deviation is known (100 hours). The z-statistic counts how many standard errors the sample mean is from the hypothesised mean, and the p-value comes from the standard normal distribution.

The alternative: when the population standard deviation is unknown (the usual situation in practice), estimate it from the sample and use a **t-test**, which accounts for that extra uncertainty.

In [ ]:
z_statistic = (sample_mean - population_mean) / standard_error
p_value = 2 * (1 - norm.cdf(abs(z_statistic)))

print(f"Sample Mean: {sample_mean}")
print(f"Z-Statistic: {z_statistic}")
print(f"P-Value: {p_value}")

The sample mean comes out at about 977.5 hours, a z-statistic of about -1.59: the sample mean sits 1.59 standard errors below 1000. The p-value is about 0.11, meaning that if the true mean really were 1000 hours, a deviation this large or larger would still occur in roughly 11% of samples. That is not unusual enough to be convincing evidence.

### Step 5: Make the decision

In [ ]:
alpha = 0.05
if p_value < alpha:
    print("Reject the null hypothesis.")
else:
    print("Fail to reject the null hypothesis.")

### Interpretation

With p = 0.11 > alpha = 0.05, we **fail to reject** the null hypothesis: this sample gives no significant evidence that the mean lifetime differs from 1000 hours.

Note the wording. Failing to reject is not proof that the mean is exactly 1000 hours; it means the data is compatible with 1000 hours. Had the p-value been below alpha, we would have concluded the mean lifetime differs significantly from 1000 hours.

---

## 3. Case Study 2: Binomial Distribution

**Problem.** You are testing a new drug. In clinical trials, the drug is expected to have a 60% success rate. You test the drug on 20 patients and observe 15 successes. Test whether the success rate is significantly different from 60%.

### Step 1: Set up the data

Here the data is not a set of measurements but a **count of successes out of a fixed number of trials**, which is exactly what the binomial distribution describes.

In [ ]:
n_trials = 20  # Number of trials
successes = 15  # Observed successes
success_rate_hypothesis = 0.60  # Hypothesized success rate

### Step 2: State the hypotheses

- H0: p = 0.60 (the success rate is 60%)
- H1: p != 0.60 (the success rate is different from 60%)

### Step 3: Perform the z-test for a proportion

`proportions_ztest` from statsmodels approximates the binomial distribution with a normal one, which is reasonable here because n * p and n * (1 - p) are both comfortably above 5. It compares the observed proportion (15/20 = 0.75) with the hypothesised 0.60.

The alternative: with very small samples the normal approximation breaks down, and an **exact binomial test** (`scipy.stats.binomtest`) computes the p-value directly from binomial probabilities.

In [ ]:
z_stat, p_value = proportions_ztest(successes, n_trials, value=success_rate_hypothesis)

print(f"Z-Statistic: {z_stat}")
print(f"P-Value: {p_value}")

The observed success rate is 0.75 against a hypothesised 0.60, giving a z-statistic of about 1.55 and a p-value of about 0.12. Fifteen successes out of 20 sounds impressive, but with only 20 patients the proportion estimate is noisy: a true 60% drug would produce 15 or more (or 9 or fewer) successes more than one time in ten.

### Step 4: Make the decision

In [ ]:
alpha = 0.05
if p_value < alpha:
    print("Reject the null hypothesis.")
else:
    print("Fail to reject the null hypothesis.")

### Interpretation

With p = 0.12 > alpha = 0.05, we **fail to reject** the null hypothesis: the trial does not provide significant evidence that the success rate differs from 60%. A larger trial would be needed to distinguish a genuine improvement from sampling noise.

---

## 4. Case Study 3: Poisson Distribution

**Problem.** You work for a customer support centre. On average, 10 calls come in per hour. You want to test whether the average number of calls per hour has changed. You observe 8 calls in the last hour.

### Step 1: State the hypotheses

- H0: lambda = 10 (the average number of calls is 10 per hour)
- H1: lambda != 10 (the average number of calls is different from 10 per hour)

### Step 2: Set up the observed and expected counts

Event counts per fixed interval (calls per hour) follow a **Poisson distribution**. With a single observed count there is no sample mean or standard error to compute; the observation itself is the test statistic.

In [ ]:
# For Poisson distribution, the test statistic is based on the observed count and the expected count.
observed_calls = 8
expected_calls = 10

### Step 3: Calculate the p-value from the Poisson distribution

Instead of approximating with a normal curve, we use the Poisson probabilities directly: this is an **exact test**. For a two-sided test we take the smaller tail probability (here, observing 8 or fewer calls when 10 are expected) and double it.

The alternative: with many observed hours rather than a single one, the total count grows large and a normal approximation (or a z-test on the hourly mean) becomes reasonable.

In [ ]:
# Since it's a two-tailed test, we calculate the cumulative probability on both sides of the observed value
p_value = 2 * min(
    poisson.cdf(observed_calls, expected_calls),
    1 - poisson.cdf(observed_calls - 1, expected_calls),
)

print(f"Observed Calls: {observed_calls}")
print(f"Expected Calls: {expected_calls}")
print(f"P-Value: {p_value}")

The p-value is about 0.67: if calls really arrive at 10 per hour, observing a count at least as far from 10 as ours is very common. A single quiet hour with 8 calls is entirely unremarkable, since the Poisson standard deviation at lambda = 10 is sqrt(10), about 3.2 calls.

### Step 4: Make the decision

In [ ]:
alpha = 0.05
if p_value < alpha:
    print("Reject the null hypothesis.")
else:
    print("Fail to reject the null hypothesis.")

### Interpretation

With p = 0.67 > alpha = 0.05, we **fail to reject** the null hypothesis: one hour with 8 calls gives no evidence that the call rate has changed from 10 per hour. Detecting a genuine change in a rate requires observing more intervals, not staring harder at one.

---

## Summary

In this notebook you:

- Stated null and alternative hypotheses for three business questions.
- Selected the test to match the data's distribution: a z-test for a normal mean with known standard deviation, a proportion z-test for binomial successes, and an exact test for a Poisson count.
- Computed test statistics and p-values with scipy and statsmodels.
- Applied the alpha = 0.05 decision rule and interpreted each outcome in context.

| Case study | Distribution | Test | p-value | Decision at alpha = 0.05 |
|---|---|---|---|---|
| Light bulb lifetimes | Normal | z-test (known std) | 0.11 | Fail to reject H0 |
| Drug success rate | Binomial | Proportion z-test | 0.12 | Fail to reject H0 |
| Support calls per hour | Poisson | Exact Poisson test | 0.67 | Fail to reject H0 |

All three tests fail to reject: in each case the sample was too small (or the deviation too mild) to rule out chance. That is the everyday reality of hypothesis testing, and it is why sample size matters as much as the test you choose.

## References & Further Reading

- [**scipy.stats reference**](https://docs.scipy.org/doc/scipy/reference/stats.html): The distributions and tests used in this notebook.
- [**statsmodels proportions_ztest**](https://www.statsmodels.org/stable/generated/statsmodels.stats.proportion.proportions_ztest.html): Documentation for the proportion z-test.
- [**StatQuest: Hypothesis Testing and The Null Hypothesis**](https://www.youtube.com/watch?v=0oc49DyA3hU): The core idea in 15 minutes.
- [**StatQuest: p-values, what they are and how to interpret them**](https://www.youtube.com/watch?v=vemZtEM63GY): What a p-value does and does not tell you.
- [**ModernDive: Hypothesis Testing**](https://moderndive.com/v2/hypothesis-testing.html): A simulation-first treatment of the same ideas.